# 01. 칵테일 무드태그 사전 임베딩

**목적**: `cocktails_final.csv`의 `mood_tag` 컬럼을 `nomic-embed-text`로 임베딩하여 `.npy` 파일로 저장.

**실행 시점**: 딱 한 번만 실행하면 됨. 이후 `02_recommend.ipynb`에서 불러와 사용.

**출력 파일**:
- `cocktail_embeddings.npy` — 100개 칵테일 임베딩 벡터 (shape: 100 x 768)
- `cocktail_ids.npy` — 각 벡터에 대응하는 cocktail_id 순서

In [1]:
import pandas as pd
import numpy as np
import requests
import os
from tqdm import tqdm

In [2]:
# 경로 설정
CSV_PATH = "/home/piai/b2/PBA_AI_Project/modeling/data/cocktails_final.csv"
SAVE_DIR = "/home/piai/b2/PBA_AI_Project/modeling/image/final"
EMBED_MODEL = "nomic-embed-text"
OLLAMA_URL = "http://localhost:11434/api/embeddings"

df = pd.read_csv(CSV_PATH)
print(f"칵테일 수: {len(df)}")
print(df[["cocktail_id", "name_kr", "mood_tag"]].head())

칵테일 수: 101
   cocktail_id      name_kr              mood_tag
0            1      그랑 마가리타    social|cool|casual
1            2         마가리타    social|cool|casual
2            3     토미스 마가리타    social|cool|casual
3            4  네이키드 앤 페이머스  relaxing|dark|modern
4            5     데킬라 선라이즈    cozy|bright|casual


In [3]:
def get_embedding(text: str) -> list:
    """nomic-embed-text로 텍스트 임베딩"""
    response = requests.post(
        OLLAMA_URL,
        json={"model": EMBED_MODEL, "prompt": text}
    )
    response.raise_for_status()
    return response.json()["embedding"]

# 연결 테스트
test = get_embedding("cozy warm private")
print(f"임베딩 차원: {len(test)}")

임베딩 차원: 768


In [4]:
# mood_tag를 파이프(|) 제거 후 공백으로 연결
# 예: "social|cool|casual" → "social cool casual"
df["mood_tag_clean"] = df["mood_tag"].str.replace("|", " ", regex=False)

embeddings = []
cocktail_ids = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="임베딩 중"):
    emb = get_embedding(row["mood_tag_clean"])
    embeddings.append(emb)
    cocktail_ids.append(row["cocktail_id"])

embeddings = np.array(embeddings)
cocktail_ids = np.array(cocktail_ids)

print(f"임베딩 완료: {embeddings.shape}")

임베딩 중: 100%|██████████| 101/101 [00:07<00:00, 14.31it/s]

임베딩 완료: (101, 768)


In [5]:
# 저장
np.save(os.path.join(SAVE_DIR, "cocktail_embeddings.npy"), embeddings)
np.save(os.path.join(SAVE_DIR, "cocktail_ids.npy"), cocktail_ids)

print(f"저장 완료")
print(f"  cocktail_embeddings.npy: {embeddings.shape}")
print(f"  cocktail_ids.npy: {cocktail_ids.shape}")

저장 완료
  cocktail_embeddings.npy: (101, 768)
  cocktail_ids.npy: (101,)
